# 01 — Standalone Inference

Smallest possible end-to-end example: load the exported FastChem emulator
bundle and predict mixing ratios on a synthetic isothermal profile.

The bundle is weights + JSON metadata only; the forward pass lives in
`src/models/transformer.py` and is reached through
`src.models.standalone_inference`. There are no ExoJAX or ExoGibbs imports
in this notebook — it is the cleanest demonstration of what the emulator
needs and what it returns.


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

# Resolve the project root whether this notebook is launched from
# `exojax_demo/` or from the repo root.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "exojax_demo":
    PROJECT_ROOT = PROJECT_ROOT.parent

# Pick the bundle to load. Override via the VULCAN_DEMO_MODEL env var
# (used by the PBS submission script in supercomputer_cmds/).
import os
MODEL = os.environ.get("VULCAN_DEMO_MODEL", "fastchem")
BUNDLE_PATH = (PROJECT_ROOT / "models" / MODEL / "best_exported.npz").resolve()
assert BUNDLE_PATH.exists(), f"bundle not found at {BUNDLE_PATH}"

# Put the distribution root on sys.path so `src` imports resolve.
DIST_ROOT = BUNDLE_PATH.parents[2]
assert (DIST_ROOT / "src").is_dir(), (
    f"src not found at {DIST_ROOT / 'src'} — the distribution must keep "
    "`src/` next to `models/`."
)
if str(DIST_ROOT) not in sys.path:
    sys.path.insert(0, str(DIST_ROOT))

# Project-shipped matplotlib style (ships in this folder).
_STYLE = Path("science.mplstyle")
if _STYLE.exists():
    plt.style.use(str(_STYLE))

print(f"MODEL       : {MODEL}")
print(f"BUNDLE_PATH : {BUNDLE_PATH}")


## Load the bundle

`load_model` returns an `ExportedModel` that exposes the data contract,
species labels, and a `predict_fastchem_profile(...)` method.


In [ ]:
from src.models.standalone_inference import load_model

model = load_model(BUNDLE_PATH)
print(f"chemistry      : {model.chemistry_type}")
print(f"model type     : {model.model_type}")
print(f"output species : {model.species}")
print(f"global order   : {model.data_contract['global_static_feature_order']}")
print(f"levels range   : {tuple(model.data_contract['num_levels_range'])}")
print(f"pressure span  : {model.data_contract['pressure_top_bar_range']} -> "
      f"{model.data_contract['pressure_bottom_bar_range']} bar")


## Predict on a synthetic profile

We use the training-anchor solar abundances from `src.constants.SOLAR_ABUNDANCES`
so the demo lands at the centre of the training distribution. A 50-level grid
spanning 100 → 1e-6 bar comfortably sits inside `num_levels_range` and the
trained pressure span.


In [ ]:
from src.constants import SOLAR_ABUNDANCES

pressure_bar = np.logspace(2.0, -6.0, 50)
temperature_k = np.full_like(pressure_bar, 1500.0)
global_inputs = {key: float(value) for key, value in SOLAR_ABUNDANCES.items()}

predictions_log10 = np.asarray(
    model.predict_fastchem_profile(
        pressure_bar=pressure_bar,
        temperature_k=temperature_k,
        global_inputs=global_inputs,
        return_log10=True,
    )
)

phot = int(np.argmin(np.abs(pressure_bar - 0.1)))
print(f"output shape : {predictions_log10.shape}")
print(f"log10 VMR @ P = {pressure_bar[phot]:.3f} bar:")
for name in ("H2", "He", "H2O", "CO", "CO2", "CH4", "NH3", "H2S"):
    if name in model.species:
        col = model.species.index(name)
        print(f"  {name:>5s}  log10(ymix) = {predictions_log10[phot, col]:+.3f}")


## Plot the predicted profile

Mixing ratio vs pressure for the 17 trained species, on the same isothermal
T(P) used above. This is a sanity-check plot — see `02_chemistry_comparison`
for apples-to-apples overlays against FastChem and ExoGibbs.


In [ ]:
ymix = np.power(10.0, predictions_log10)

fig, ax = plt.subplots(figsize=(6.5, 7))
colors = plt.cm.tab20(np.linspace(0.0, 1.0, len(model.species)))
for i, name in enumerate(model.species):
    ax.plot(np.clip(ymix[:, i], 1e-30, None), pressure_bar,
            color=colors[i], lw=1.4, label=name)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlim(1.0e-20, 3.0)
ax.invert_yaxis()
ax.set_xlabel("Mixing ratio")
ax.set_ylabel("Pressure (bar)")
ax.set_title(f"Emulator @ T = 1500 K (solar X/H), {MODEL}")
ax.legend(fontsize=7, ncol=3, loc="lower left")
plt.tight_layout()
plt.show()
